this is the bulk of the computing, and it's "results" are all stored (except the datasets) and pushed so that it's not necessary to rerun

In [ ]:
# datagen
import sys

script = "../scripts/generate_data.py"
!"{sys.executable}" -u {script}

In [ ]:
# training the models
import sys

script = "../scripts/train_resnet.py"
!"{sys.executable}" -u {script} --CfgModel.n-blocks-per-stage  3 --CfgModel.use-skip 
!"{sys.executable}" -u {script} --CfgModel.n-blocks-per-stage  3 --CfgModel.no-use-skip
!"{sys.executable}" -u {script} --CfgModel.n-blocks-per-stage 12 --CfgModel.use-skip 
!"{sys.executable}" -u {script} --CfgModel.n-blocks-per-stage 12 --CfgModel.no-use-skip

In [ ]:
# compute and store inference (LL)
from pathlib import Path
from loss_landscapes.paths import PROJ_ROOT
import subprocess, sys

checkpoints = sorted((PROJ_ROOT / "checkpoints").glob("*.pt"))

def run(script, *args):
    proc = subprocess.Popen(
        [sys.executable, "-u", script, *args],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"script failed: {script} {args}")

for ckpt in checkpoints:
    for method in ["naive", "filternorm"]:
        print(f"===== {ckpt.name} | {method} =====")
        run("../scripts/generate_landscape.py", "--checkpoint", ckpt.name, "--method", method, "--npoints", "101")